# Lahore Land Surface Temperature (Landsat 8/9)
## Export: 100 m temperature GeoTIFF and 100 m point GeoJSON


### 0. Initialize Earth Engine


In [2]:
import calendar
import urllib.request

import ee, geemap, geopandas as gpd

try:
    ee.Initialize()
except Exception:
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize()


### 1. Parameters


In [3]:
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"
YEAR = 2026
MONTH = 2
last_day = calendar.monthrange(YEAR, MONTH)[1]
START = f"{YEAR:04d}-{MONTH:02d}-01"
END = f"{YEAR:04d}-{MONTH:02d}-{last_day:02d}"

RASTER_SCALE = 100
POINT_SCALE = 100
OUT_PREFIX = f"LST_Lahore_{calendar.month_abbr[MONTH]}{YEAR}"

print(f"Using date range: {START} to {END}")
print(f"Output prefix: {OUT_PREFIX}")

Using date range: 2026-02-01 to 2026-02-28
Output prefix: LST_Lahore_Feb2026


### 2. Load Lahore Boundary and Build Monthly LST Composite


In [4]:
gdf = gpd.read_file(UC_SHP)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()

COLS = ["LANDSAT/LC08/C02/T1_L2", "LANDSAT/LC09/C02/T1_L2"]

def mask_l2(img):
    qa = img.select("QA_PIXEL")
    mask = (
        qa.bitwiseAnd(1 << 3).eq(0)
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
        .And(qa.bitwiseAnd(1 << 7).eq(0))
    )
    return img.updateMask(mask)

def st_to_c(img):
    st = img.select("ST_B10")
    st = st.multiply(0.00341802).add(149.0)
    st_c = st.subtract(273.15).rename("LST_C")
    return img.addBands(st_c, overwrite=True)

def prep(img):
    return st_to_c(mask_l2(img)).select("LST_C").clip(region)

col = (
    ee.ImageCollection(COLS[0])
    .merge(ee.ImageCollection(COLS[1]))
    .filterBounds(region)
    .filterDate(START, END)
    .map(prep)
)

lst_c_med = col.median().rename("LST_C")


### 3. Export GeoTIFF and Point GeoJSON


In [5]:
def export_image_local(img, filename, region, scale=100, crs="EPSG:4326"):
    try:
        geemap.ee_export_image(img.clip(region), filename, scale=scale, region=region, file_per_band=False)
        print(f"[OK] GeoTIFF -> {filename}")
        return
    except TypeError:
        pass
    except Exception as e:
        print("[INFO] ee_export_image failed:", e)
    try:
        geemap.download_ee_image(img.clip(region), filename=filename, scale=scale, region=region, crs=crs)
        print(f"[OK] GeoTIFF (download_ee_image) -> {filename}")
        return
    except Exception as e:
        print("[INFO] download_ee_image failed:", e)
    url = img.clip(region).getDownloadURL({
        "scale": scale,
        "crs": crs,
        "region": region,
        "filePerBand": False,
    })
    urllib.request.urlretrieve(url, filename)
    print(f"[OK] GeoTIFF (getDownloadURL) -> {filename}")

tif_path = f"{OUT_PREFIX}_{RASTER_SCALE}m.tif"
export_image_local(lst_c_med, tif_path, region, scale=RASTER_SCALE)

pts_fc = ee.Image.pixelLonLat().addBands(lst_c_med.rename("val")).sample(
    region=region,
    scale=POINT_SCALE,
    geometries=True,
    seed=1,
)

geojson_pts = f"{OUT_PREFIX}_points_{POINT_SCALE}m.geojson"
geemap.ee_export_vector(pts_fc, filename=geojson_pts)
print(f"[OK] GeoJSON points -> {geojson_pts}")

Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/LST/LST_Lahore_Feb2026_100m.tif
[OK] GeoTIFF -> LST_Lahore_Feb2026_100m.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/LST/LST_Lahore_Feb2026_points_100m.geojson
[OK] GeoJSON points -> LST_Lahore_Feb2026_points_100m.geojson
